# eManual Experiment Runner — OpenRouter variant (parallel to the Groq run)

Same emanual RAG sweep as `emanual_experiment_notebook.ipynb`, but all LLM calls
go through **OpenRouter** instead of Groq. Run this notebook alongside the Groq
one to parallelize without sharing Groq's rate limit.

Driven by `experiment_configs/emanual_openrouter_experiment.yaml` and the configs
under `rag-experiments/emanual-openrouter-experiment/config/`. It uses its own
`temp/`, `reports/`, and `cache_openrouter/` dirs so the two runs never collide.

**Requires** `OPENROUTER_API_KEY` (comma-separate multiple keys to rotate). The
`openai` package must be installed (it is added to the install cell below).


## 1. Setup & Dependencies

In [8]:
get_ipython().system('pip3 install datasets faiss-cpu sentence-transformers torch groq openai python-dotenv nltk pandas -q')


## 2. Imports

In [9]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# --- Point this at wherever THIS repo (rag_cust_support) lives. ---
# On Colab this is typically under your mounted Drive. Adjust if different.
PROJECT_ROOT = Path('/content/drive/MyDrive/Capstone/rag_cust_support')
if not PROJECT_ROOT.exists():
    # Fallback: running locally from the notebooks/ folder.
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()

os.chdir(PROJECT_ROOT)
project_root = PROJECT_ROOT
# Make THIS repo win on sys.path (avoids importing a stale rag-foundry copy).
sys.path = [p for p in sys.path if 'rag-foundry' not in p]
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))

from experiment.experiment_config import ExperimentConfig
from experiment.experiment_runner import ExperimentRunner
import experiment.experiment_runner as _er, core.registry as _reg

load_dotenv(override=True)
print('Current directory:', Path.cwd())
print('experiment_runner loaded from:', _er.__file__)
print('core.registry   loaded from:', _reg.__file__)
assert 'rag-foundry' not in _er.__file__, 'Still importing the old rag-foundry code! Restart runtime.'
print('HuggingFace token loaded:', bool(os.getenv('HF_TOKEN')))
print('Groq API key loaded:', bool(os.getenv('GROQ_API_KEY')))
print('OpenRouter API key loaded:', bool(os.getenv('OPENROUTER_API_KEY')))


Current directory: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support
experiment_runner loaded from: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support/experiment/experiment_runner.py
core.registry   loaded from: /Users/bhupendra.bhoi/aiml/Capstone Project/rag_cust_support/core/registry.py
HuggingFace token loaded: True
Groq API key loaded: True
OpenRouter API key loaded: True


## 3. Load Experiment Configuration

The experiment configuration file specifies:
- **data_loader**: How to load data (HuggingFace with dataset_name, subset, split)
- **data_parser**: How to parse documents (title_passage)
- **config_dir**: Directory containing RAG pipeline configs
- **num_queries**: Number of queries to evaluate
- **parallel**: Whether to run configs in parallel

In [10]:
EXPERIMENT_CONFIG_PATH = project_root / "experiment_configs/emanual_openrouter_experiment.yaml"

experiment_config = ExperimentConfig.load(EXPERIMENT_CONFIG_PATH)

print("Experiment Configuration:")
print(f"  Config Dir:  {experiment_config.config_dir}")
print(f"  Report Dir:  {experiment_config.report_dir}")
print(f"  Temp Dir:    {experiment_config.temp_dir}")
print(f"  Cache:       {experiment_config.cache}")
print(f"  Num Queries: {experiment_config.end_index}")
print(f"  Parallel:    {experiment_config.parallel}")
print(f"  Max Workers: {experiment_config.max_workers}")
print(f"\nData Loader:")
print(f"  Type: {experiment_config.data_loader['type']}")
print(f"  Config: {experiment_config.data_loader['config']}")
print(f"\nData Parser:")
print(f"  Type: {experiment_config.data_parser}")

Experiment Configuration:
  Config Dir:  rag-experiments/emanual-openrouter-experiment/config
  Report Dir:  rag-experiments/emanual-openrouter-experiment/reports
  Temp Dir:    rag-experiments/emanual-openrouter-experiment/temp
  Cache:       {'enabled': True, 'cache_dir': './cache_openrouter'}
  Num Queries: 20
  Parallel:    False
  Max Workers: 1

Data Loader:
  Type: huggingface
  Config: {'dataset_name': 'galileo-ai/ragbench', 'subset': 'emanual', 'split': 'test', 'limit': 132}

Data Parser:
  Type: noop


## 4. Initialize Experiment Runner

The ExperimentRunner will:
- Create report directory if it doesn't exist
- Load RAG configs from the specified directory
- Load and parse data automatically based on YAML config

In [11]:
# Initialize experiment runner
runner = ExperimentRunner(experiment_config)
print("ExperimentRunner initialized")

ExperimentRunner initialized


In [12]:
# Load data automatically based on YAML configuration
print("Loading data based on experiment configuration...")
documents, raw_data = runner.load_data()

print(f"\n✅ Data loaded successfully!")
print(f"  Documents: {len(documents)} parsed documents")
print(f"  Raw Data:  {len(raw_data)} samples")

# Inspect first sample
first_sample = raw_data[0]
print(f"\nFirst Sample:")
print(f"  Question: {first_sample['question'][:100]}...")
print(f"  Documents: {len(first_sample['documents'])}")

Loading data based on experiment configuration...
Loading HuggingFace dataset: galileo-ai/ragbench/emanual (test)...
Loaded 132 samples

✅ Data loaded successfully!
  Documents: 101 parsed documents
  Raw Data:  132 samples

First Sample:
  Question: I want to  enter into Ambient mode. How can I do that?...
  Documents: 3


## 6. Load RAG Pipeline Configs

Load all RAG pipeline configurations from the config directory specified in the experiment config.

In [13]:
# Load RAG pipeline configs
configs = runner.load_configs()

print(f'Loaded {len(configs)} RAG pipeline configurations:')
for cfg in configs:
    searches = ' + '.join(s.type.value for s in cfg.retrieval.search.searches)
    fusion = cfg.retrieval.fusion.type.value if cfg.retrieval.fusion else '-'
    rerank = cfg.retrieval.rerank.type.value if cfg.retrieval.rerank else '-'
    qx = cfg.retrieval.query_transform.type.value if cfg.retrieval.query_transform else '-'
    gc = cfg.generation.config
    model = gc.get('model') if isinstance(gc, dict) else getattr(gc, 'model', None)
    print(f'  - {cfg.name}')
    print(f'      chunking={cfg.chunking.type.value}  embed={cfg.embedding.type.value}')
    print(f'      search=[{searches}]  fusion={fusion}  rerank={rerank}  q_transform={qx}')
    print(f'      generation_model={model}')


Loaded 8 RAG pipeline configurations:
  - emanual_or_v1_baseline
      chunking=fixed_word  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=meta-llama/llama-3.3-70b-instruct
  - emanual_or_v2_hybrid_wsum
      chunking=fixed_word  embed=sentence_transformer
      search=[dense + sparse]  fusion=weighted_sum  rerank=-  q_transform=-
      generation_model=meta-llama/llama-3.3-70b-instruct
  - emanual_or_v3_embed_bge
      chunking=fixed_word  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=meta-llama/llama-3.3-70b-instruct
  - emanual_or_v4_chunk_sentence
      chunking=sentence  embed=sentence_transformer
      search=[dense]  fusion=-  rerank=-  q_transform=-
      generation_model=meta-llama/llama-3.3-70b-instruct
  - emanual_or_v5_hybrid_rrf
      chunking=fixed_word  embed=sentence_transformer
      search=[dense + sparse]  fusion=rrf  rerank=-  q_transform=-
      ge

## 7. Run Experiments

Run all RAG configurations on the loaded data. Each config will:
1. Build a vector index from the documents
2. Run queries against the index
3. Generate responses
4. Evaluate using TRACe metrics

Results are returned as PipelineRunResult objects.

In [15]:
get_ipython().system('pip install rank_bm25 -q')

# Run experiments

In [14]:
# Run experiments

print(f"Running {len(configs)} configurations from {experiment_config.start_index} to {experiment_config.end_index} queries...")
print(f"Parallel mode: {experiment_config.parallel}")

runs = runner.run(documents, raw_data)

print(f"\n✅ Experiments completed!")
print(f"  Ran {len(runs)} configurations")

for run in runs:
    print(f"  - {run['config'].name}: {run['total_written']} queries")

Running 8 configurations from 0 to 20 queries...
Parallel mode: False
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15902.14it/s]


Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 0sUsing key #0: ****f2e4Using key #0: ****f2e4

Progress: 0/20 (0.0%) | QPS: 0.00 | ETA: Unknown | Elapsed: 3sUsing key #0: ****f2e4
Using key #0: ****f2e4
Progress: 2/20 (10.0%) | QPS: 0.22 | ETA: 1.4m | Elapsed: 9sUsing key #0: ****f2e4
Progress: 3/20 (15.0%) | QPS: 0.30 | ETA: 57s | Elapsed: 10sUsing key #0: ****f2e4
Progress: 5/20 (25.0%) | QPS: 0.23 | ETA: 1.1m | Elapsed: 22sUsing key #0: ****f2e4
Progress: 6/20 (30.0%) | QPS: 0.17 | ETA: 1.4m | Elapsed: 36sUsing key #0: ****f2e4
Progress: 6/20 (30.0%) | QPS: 0.16 | ETA: 1.5m | Elapsed: 38sUsing key #0: ****f2e4
Progress: 7/20 (35.0%) | QPS: 0.17 | ETA: 1.2m | Elapsed: 40sUsing key #0: ****f2e4
Progress: 8/20 (40.0%) | QPS: 0.19 | ETA: 1.1m | Elapsed: 42sUsing key #0: ****f2e4
Progress: 9/20 (45.0%) | QPS: 0.21 | ETA: 53s | Elapsed: 43sUsing key #0: ****f2e4
Progress: 10/20 (50.0%) | QPS: 0.22 | ETA: 46s | Elapsed: 46sUsing key #0: ****f2e4
Progress: 11/20 (55.0%) | QPS: 

ModuleNotFoundError: No module named 'rank_bm25'

## 7b. Evaluate Existing JSONL Files

Run offline evaluation on already-generated JSONL files.
Uses experiment-level evaluation config — all configs are scored with the same judge model.

- `parallel_runs=True` — evaluate multiple configs simultaneously
- `parallel_config_run=True` — evaluate records within each config in parallel

In [ ]:
# Discover all configs and build run dicts from existing JSONL files
configs = runner.load_configs()
runs = []
for cfg in configs:
    jsonl_path = experiment_config.temp_dir / f"{cfg.name}.jsonl"
    if jsonl_path.exists():
        runs.append({"config_name": cfg.name, "config": cfg, "jsonl_path": jsonl_path})
    else:
        print(f"  Skipping {cfg.name} — no JSONL found")

print(f"Found {len(runs)} configs with JSONL files")

# Evaluate all configs: parallel across configs + parallel within each config
eval_runs = runner.evaluate_runs(
    runs,
    parallel_runs=True,
    parallel_config_run=True,
)

# Use eval_runs for report generation downstream
runs = eval_runs
print(f"\n✅ Evaluation complete: {len(eval_runs)} configs")

## 8. Generate Reports

Generate detailed reports for each configuration including:
- Per-query table with all TRACe scores
- Aggregate statistics (mean, std, MAE)
- Comparison with ground truth

In [ ]:
# Generate reports
print("Generating reports...")
reports = runner.generate_reports(runs)

print(f"\n✅ Reports generated!")
print(f"  Saved to: {experiment_config.report_dir}")

## 9. Display Reports

Display the generated reports with per-query and aggregate metrics.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

for report in reports:
    print(f"\n{'='*80}")
    print(f"Configuration: {report.config_name}")
    print(f"{'='*80}")
    
    # Display per-query results
    print("\nPer-Query Results:")
    display(report.display())
    

## 10. Compare Configurations

Generate a comparison report across all configurations to see which performs best.

In [ ]:
# Generate comparison report
print("Generating comparison report...")
comparison = runner.compare()

print(f"\n✅ Comparison report generated!")
print(f"  Saved to: {experiment_config.report_dir}/comparison.csv")

In [ ]:
# Display comparison
print("\nConfiguration Comparison:")
display(comparison.to_dataframe())

## 11. Summary

The ExperimentRunner provides a complete workflow for:

1. **Configuration-driven data loading** - Specify data source in YAML
2. **Automatic parsing** - Documents parsed using configured parser
3. **Multi-config evaluation** - Test multiple RAG configurations
4. **Parallel execution** - Speed up evaluation with parallel runs
5. **Comprehensive reporting** - Per-query and aggregate metrics
6. **Cross-config comparison** - Identify best performing config

### Key Benefits:

- **Reproducible** - Everything configured in YAML
- **Flexible** - Easy to change data source or parser
- **Scalable** - Parallel execution for faster evaluation
- **Comprehensive** - Detailed metrics and comparisons